# Evaluation

This notebook evaluates the four models from the modeling notebook on the test set: 21,781 crashes (20%) that the models never saw during training. The code is in `src/data3800_gruppe_7/evaluation.py`.

Run the modeling notebook or `uv run python -m data3800_gruppe_7.modeling` first, so that the models and predictions exist in `output/models/`. Running `uv run python -m data3800_gruppe_7.evaluation` saves the same tables and figures in `output/evaluation/`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from data3800_gruppe_7.evaluation import (
    bootstrap_intervals,
    compute_metrics,
    compute_permutation_importance,
    load_models,
    load_predictions,
    load_test_set,
    logistic_odds_ratios,
    plot_confusion_matrices,
    plot_curves,
    plot_permutation_importance,
)

In [ ]:
predictions = load_predictions()
pipelines = load_models(predictions["model"].unique())
X_test, y_test = load_test_set(predictions)
print(f"{len(y_test)} test crashes, {y_test.mean():.1%} with at least one injury")

## Metrics

Injury is the positive class. With TP, FP, FN and TN as the cells of the confusion matrix:

| Metric | Formula | Question it answers |
| --- | --- | --- |
| Accuracy | (TP + TN) / all | How many crashes are classified correctly? |
| Precision | TP / (TP + FP) | Of the crashes flagged as injury, how many had one? This is P(injury \| predicted injury). |
| Recall | TP / (TP + FN) | Of the crashes with an injury, how many were flagged? |
| F1 | 2 · precision · recall / (precision + recall) | The harmonic mean of the two, high only when both are. |
| ROC-AUC | area under the ROC curve | The chance that a random injury crash gets a higher probability than a random crash without one. 0.5 is a coin flip. |
| PR-AUC | average precision | Precision averaged over all recall levels. A model that guesses at random gets the base rate, 0.164. |

The first four use the predicted class, which is injury when the predicted probability is at least 0.5. The two AUCs use the probability itself and do not depend on that threshold. Because only 16.4% of the crashes had an injury, the EDA concluded that accuracy says little here and that we should focus on recall, precision, F1 and PR-AUC.

In [ ]:
metrics = compute_metrics(predictions)
metrics.round(3)

**Accuracy ranks the models the wrong way round.** The baseline has the highest accuracy, 83.6%, because it predicts "no injury" for every crash, and it finds none of the injuries. The three real models have accuracy between 42 and 48%, since the balanced class weights make them flag 60 to 68% of the crashes as injury.

**The models find most injuries, but at a high cost in false alarms.** Recall is 0.74 to 0.81, while precision is only about 0.20. Of the crashes a model flags, one in five had an injury, compared with one in six (16.4%) for a crash picked at random. Following Bayes' theorem as in the modeling notebook: the conditions raise the probability of injury from the base rate of 0.164 to about 0.20 among the flagged crashes, a lift of only about 1.2.

**The ranking metrics agree.** ROC-AUC is 0.60 to 0.61 and PR-AUC 0.21 to 0.22, clearly better than chance (0.5 and 0.164) but far from a model that separates the classes well. The five recorded conditions carry real but weak information about whether a crash causes an injury.

## Are the differences between the models real?

The test set is one sample of crashes, so the metrics have sampling noise. We draw 1,000 bootstrap resamples of the test set and recompute F1 and PR-AUC on each, giving 95% percentile intervals. Every resample uses the same crashes for all models, so we can also put an interval around the difference between two models. If the interval of a difference contains 0, the test set cannot tell the two models apart.

In [ ]:
intervals = bootstrap_intervals(predictions)
intervals.round(4)

- **Logistic regression and the random forest are equally good.** The interval of their difference in PR-AUC is about [-0.007, 0.003] and contains 0, and the same holds for F1.
- **The decision tree ranks crashes worse.** Its PR-AUC is lower than both other models, and neither interval contains 0. Its F1 is not clearly lower, since it makes up for the lower precision with the highest recall.

Logistic regression is therefore the model we prefer. It is as good as the random forest, much simpler, and its coefficients can be read directly, which we use below. A random forest can pick up combinations of conditions that a linear model misses, so its lack of gain suggests that there are few such interactions to find in these five columns.

## Confusion matrices

The matrices are normalized by row, so each row shows how the crashes of one true class were classified. The diagonal is the recall of each class: the bottom right cell is the recall for injury, the top left the share of crashes without injury that were correctly left unflagged.

In [ ]:
plot_confusion_matrices(predictions);

The baseline puts every crash in the left column. The three models trade the two kinds of error differently: the decision tree finds the most injuries (81%) but flags 65% of the crashes without one, while logistic regression finds 74% and flags 58%. None of them reaches a point where both kinds of error are low, which is what the low AUC values mean in practice.

Which trade-off is best depends on the use. A screening tool that must not miss injuries would accept many false alarms, while a tool that sends resources to flagged crashes would need higher precision. Since the 0.5 threshold is not a deliberate choice, the curves below show every threshold at once.

## ROC and precision-recall curves

In [ ]:
plot_curves(predictions);

On the ROC curve, logistic regression and the random forest lie almost on top of each other, with the decision tree slightly below. On the precision-recall curve, all three stay between 0.20 and 0.25 for most of the recall range, only a little above the base rate of 16.4%. Raising the threshold to flag fewer crashes therefore buys little precision. There is no threshold at which the models become precise, only one at which they find fewer injuries.

The decision tree has only as many distinct probabilities as it has leaves, so its curves are made of straight segments.

## Feature importance

We measure importance with **permutation importance** on the test set: we shuffle one input column, which breaks its link to injury and leaves everything else unchanged, and measure how much the PR-AUC drops. We shuffle each of the five original columns rather than the one-hot columns, so every condition gets one number and the same method works for all three models. The bars are the mean over 5 shuffles, and the lines one standard deviation.

In [ ]:
importance = compute_permutation_importance(pipelines, X_test, y_test)
plot_permutation_importance(importance);

In [ ]:
importance.pivot(index="feature", columns="model", values="mean").round(4)

- **The speed limit matters most in every model**, with a drop in PR-AUC of 0.016 to 0.019. This fits the EDA, where the injury share rose steadily from 8.8% at 20 mph or less to 22.1% at 50 mph or more, and the literature, where speed is one of the most consistent factors behind injury severity.
- **Road surface and lighting come next.** For both, part of the signal comes from the UNKNOWN category, see the odds ratios below.
- **Weather adds little** once road surface is known. The two are strongly related (rain gives wet roads), so a model can use either one, and road surface is the more direct of the two.
- **Crash hour** matters more for logistic regression than for the trees. A likely reason is that hour and lighting overlap (it is dark at night), and the trees lean on lighting for the same signal, while logistic regression spreads it over both.

All the drops are small, which fits a PR-AUC that is only about 0.05 above chance in the first place. Permutation importance also says what a model *uses*, not what causes injuries: when two columns are related, shuffling one of them hurts less because the model can fall back on the other.

## Odds ratios in the logistic regression

The coefficients of the logistic regression give the direction of each effect. The odds ratio exp(coefficient) is the factor the odds of injury are multiplied by when a crash has that category, holding the other columns fixed. For the speed limit it is per standard deviation of the training set, 5.4 mph.

We show only categories with at least 200 crashes in the test set. Rarer categories, such as `BLOWING SNOW` with 16 crashes, can get large odds ratios from a handful of crashes. The model also uses a penalty (L2 regularization) and no reference category, so the odds ratios are best compared within one column, and their size should not be read too precisely.

In [ ]:
odds_ratios = logistic_odds_ratios(pipelines["logistic_regression"], X_test)
common = odds_ratios[odds_ratios["crashes"] >= 200]
common[common["column"] != "CRASH_HOUR"].round(3)

In [ ]:
hours = odds_ratios[odds_ratios["column"] == "CRASH_HOUR"]
hour_odds = hours.assign(hour=hours["category"].astype(int)).set_index("hour")["odds_ratio"]
# Show the 24 hours as four rows of six, so the table fits on the screen
hour_odds = hour_odds.sort_index().round(2).to_numpy().reshape(4, 6)
pd.DataFrame(hour_odds, index=["00-05", "06-11", "12-17", "18-23"], columns=[f"+{i}" for i in range(6)])

- **Speed limit:** 5.4 mph more gives about 28% higher odds of injury (odds ratio 1.28).
- **Wet roads** (1.23) and **darkness on a lighted road** (1.23) raise the odds compared with the other categories of the same column. **Snow** and **ice** lower them (0.75 and 0.68), the same "winter paradox" as in the EDA, where a likely explanation is that drivers slow down when the roads are visibly slippery.
- **Hour:** the odds are highest late in the evening and at night (odds ratios of 1.19 to 1.23 at 21:00, 00:00 and 04:00) and lowest from 09:00 to 12:00 (0.81 to 0.87), the same night-time peak as in the EDA. The table reads row plus column, so the cell `12-17`, `+3` is 15:00.
- **UNKNOWN is the strongest signal in three columns**, with the lowest odds ratios for lighting (0.57), road surface (0.59) and weather (0.74). That a condition is unknown cannot itself protect anyone. More likely, minor property damage crashes are often reported with less detail, so a missing condition says something about how the crash was *reported* rather than how it happened. The models use this pattern, which inflates their scores somewhat, and it would not carry over to a setting where the conditions are always recorded.

## Conclusion

1. **All three models are better than chance, but only a little.** With ROC-AUC 0.61 and PR-AUC 0.22 against a base rate of 0.164, weather, lighting, road surface, speed limit and hour give real but weak information about whether a crash causes an injury. Most of what decides injury, such as the crash type, the speed at impact, seat belts and who was involved, is not among our inputs.
2. **Accuracy is misleading here.** The baseline has the best accuracy and is useless. We compare the models on recall, precision, F1 and PR-AUC.
3. **Logistic regression is as good as the random forest** (the interval of the difference in PR-AUC contains 0) and easier to interpret, so we prefer it. The decision tree ranks crashes worse.
4. **The speed limit is the most important condition** in every model, followed by road surface and lighting. The directions agree with the EDA: higher speed limits, wet roads, darkness and night-time go with more injuries, snow and ice with fewer.
5. **UNKNOWN conditions go with far fewer injuries**, which most likely reflects how crashes are reported, not how they happen.

## Limitations

- The threshold of 0.5 together with balanced class weights is not tuned. Choosing a threshold needs a validation set, which we do not have, and should not be done on the test set.
- The split is random. Testing on the last months of 2025 would be closer to how a model would be used, as noted in the modeling notebook.
- All results are associations in police-reported crashes. They do not show that a condition causes injuries, and they say nothing about trips that ended without a crash.